In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from attention import TinyGPT, iterate_minibatches, fetch_wikipedia_text
import torch
import torch.nn as nn
import torch.optim as optim
import requests



torch.manual_seed(42)
print(sys.executable)

c:\Users\yekta\Desktop\LLM Experiments\llm_experiments\Scripts\python.exe


In [2]:
print(torch.__version__, torch.version.cuda)
print(torch.cuda.is_available())

2.11.0+cu128 12.8
True


In [3]:
urls = [
    "https://en.wikipedia.org/wiki/Dog",
    "https://en.wikipedia.org/wiki/Wolf"
    "https://en.wikipedia.org/wiki/English_Toy_Terrier_(Black_%26_Tan)"
]

wiki_text = fetch_wikipedia_text(urls)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text_shakespeare = requests.get(url).text.lower()

text = wiki_text + " <doc> " + text_shakespeare.lower()

print("\nFinal text length:", len(text))
print(text[:1000])

text = text.lower()

# simple cleaning
text = text.replace("\n", " ")
text = text.replace(".", " .")
text = text.replace(",", " ,")


Fetched: https://en.wikipedia.org/wiki/Dog
Status: 200
Paragraphs: 79
Failed to fetch https://en.wikipedia.org/wiki/Wolfhttps://en.wikipedia.org/wiki/English_Toy_Terrier_(Black_%26_Tan): 403 Client Error: Too many requests. Please respect our robot policy https://w.wiki/4wJS. (dd12474) for url: https://en.wikipedia.org/wiki/Wolfhttps://en.wikipedia.org/wiki/English_Toy_Terrier_(Black_%26_Tan)

Final text length: 1163948
thedog(canis familiarisorcanis lupus familiaris) is adomesticateddescendant of wolves. also called thedomestic dog, it wasselectively bredduring thelate pleistocenebyhunter-gatherers. dogs and the moderngray wolfshare a common ancestor.[4]dogs were the firstspeciesto bedomesticatedover 14,000 years ago, before thedevelopment of agriculture, though genetic studies suggest the domestication process may have begun over 25,000 years ago. due to their long association with humans, dogs have gained the ability to thrive on astarch-rich diet that would be inadequate for other

In [4]:
tokens = text.split()

vocab = sorted(set(tokens))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

encoded = [stoi[w] for w in tokens]

seq_len = 64

inputs = []
targets = []

for i in range(len(encoded) - seq_len):
    inputs.append(encoded[i:i+seq_len])
    targets.append(encoded[i+1:i+seq_len+1])

input_ids = torch.tensor(inputs)
target_ids = torch.tensor(targets)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_ids = input_ids.to(device)
target_ids = target_ids.to(device)

vocab_size = len(vocab)


num_samples = input_ids.size(0)
split_idx = int(0.9 * num_samples)

train_input_ids = input_ids[:split_idx]
train_target_ids = target_ids[:split_idx]

val_input_ids = input_ids[split_idx:]
val_target_ids = target_ids[split_idx:]

print("Train shape:", train_input_ids.shape, train_target_ids.shape)
print("Val shape:  ", val_input_ids.shape, val_target_ids.shape)



model = TinyGPT(
    vocab_size=vocab_size,
    d_model=64,
    hidden_dim=64,
    num_layers=2,
    max_len=seq_len
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

batch_size = 32


In [15]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0
    train_token_correct = 0
    train_token_total = 0

    for xb, yb in iterate_minibatches(train_input_ids, train_target_ids, batch_size=batch_size, shuffle=True):
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits, _ = model(xb)  # (B, L, vocab_size)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            yb.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * xb.size(0)

        preds = logits.argmax(dim=-1)
        train_token_correct += (preds == yb).sum().item()
        train_token_total += yb.numel()

    train_loss = train_loss_sum / train_input_ids.size(0)
    train_acc = train_token_correct / train_token_total

    # validation
    model.eval()
    val_loss_sum = 0.0
    val_token_correct = 0
    val_token_total = 0

    with torch.no_grad():
        for xb, yb in iterate_minibatches(val_input_ids, val_target_ids, batch_size=batch_size, shuffle=False):
            xb = xb.to(device)
            yb = yb.to(device)
            logits, _ = model(xb)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                yb.reshape(-1)
            )

            val_loss_sum += loss.item() * xb.size(0)

            preds = logits.argmax(dim=-1)
            val_token_correct += (preds == yb).sum().item()
            val_token_total += yb.numel()

    val_loss = val_loss_sum / max(1, val_input_ids.size(0))
    val_acc = val_token_correct / max(1, val_token_total)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )



def encode_prompt(prompt_text):
    prompt_text = prompt_text.lower()
    prompt_text = prompt_text.replace("\n", " ")
    prompt_text = prompt_text.replace(".", " .")
    prompt_text = prompt_text.replace(",", " ,")
    prompt_tokens = prompt_text.split()

    prompt_ids = []
    for tok in prompt_tokens:
        if tok in stoi:
            prompt_ids.append(stoi[tok])

    return prompt_ids


def generate_text(model, prompt_text, max_new_tokens=30):
    model.eval()

    prompt_ids = encode_prompt(prompt_text)

    if len(prompt_ids) == 0:
        raise ValueError("Prompt has no known tokens in the vocabulary.")

    x = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # model max input length is seq_len here, so keep only last seq_len tokens
            x_cond = x[:, -seq_len:]

            logits, _ = model(x_cond)
            next_token_logits = logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            x = torch.cat([x, next_token_id], dim=1)

    out_tokens = [itos[idx] for idx in x[0].tolist()]
    return " ".join(out_tokens)

Epoch 01 | Train Loss: 1.1479 | Train Acc: 0.7495 | Val Loss: 26.2152 | Val Acc: 0.0639
Epoch 02 | Train Loss: 1.1233 | Train Acc: 0.7549 | Val Loss: 26.8808 | Val Acc: 0.0644
Epoch 03 | Train Loss: 1.1011 | Train Acc: 0.7598 | Val Loss: 27.3363 | Val Acc: 0.0631


In [18]:
prompt = "Maki is a dog small and cute"

In [19]:
print(generate_text(model, prompt, max_new_tokens=25))

is a dog small and a strange brooch in this hand , whose ignorant credulity will not what i think you have a doubt , and humbly stand , i
